# 02. vLLM 성능 튜닝

이 노트북에서는 vLLM의 성능 최적화 방법을 실험합니다.

## 목차
1. 설정 파라미터 이해
2. GPU Memory Utilization 튜닝
3. Batch Size 튜닝
4. 양자화 방식 비교
5. 최적 설정 찾기

## 1. 설정 파라미터 이해

vLLM의 주요 성능 관련 파라미터를 알아봅니다.

In [ ]:
import os
import time
import yaml
from dotenv import load_dotenv

load_dotenv('../.env')

# 설정 파일 로드
with open('../inference_server/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("현재 설정:")
print(yaml.dump(config, default_flow_style=False, allow_unicode=True))

In [ ]:
# 주요 파라미터 설명

parameters_explanation = {
    "gpu_memory_utilization": {
        "description": "GPU 메모리 사용 비율 (0.0 ~ 1.0)",
        "default": 0.90,
        "impact": "높을수록 더 많은 배치 처리 가능, OOM 위험 증가",
        "recommended": {
            "개발/테스트": 0.70,
            "프로덕션": 0.90,
            "최대 처리량": 0.95
        }
    },
    "max_num_batched_tokens": {
        "description": "한 번에 처리할 최대 토큰 수",
        "default": 8192,
        "impact": "높을수록 처리량 증가, 메모리 사용량 증가",
        "recommended": {
            "16GB VRAM": 4096,
            "24GB VRAM": 8192,
            "80GB VRAM": 32768
        }
    },
    "max_num_seqs": {
        "description": "동시에 처리할 최대 시퀀스 수",
        "default": 256,
        "impact": "높을수록 동시 요청 처리 가능, 지연시간 증가 가능",
        "recommended": {
            "저지연 우선": 64,
            "균형": 128,
            "처리량 우선": 256
        }
    },
    "block_size": {
        "description": "PagedAttention KV 캐시 블록 크기",
        "default": 16,
        "impact": "메모리 효율성에 영향",
        "recommended": "기본값 16 권장"
    }
}

print("주요 성능 파라미터:")
print("=" * 60)
for param, info in parameters_explanation.items():
    print(f"\n{param}:")
    print(f"  설명: {info['description']}")
    print(f"  기본값: {info['default']}")
    print(f"  영향: {info['impact']}")

## 2. GPU Memory Utilization 튜닝

GPU 메모리 사용률에 따른 성능 변화를 실험합니다.

In [ ]:
# GPU 메모리 사용률 테스트 (실제 실행에는 GPU 필요)

gpu_util_test_configs = [
    {"name": "Safe", "gpu_memory_utilization": 0.70},
    {"name": "Balanced", "gpu_memory_utilization": 0.85},
    {"name": "Production", "gpu_memory_utilization": 0.90},
    {"name": "Maximum", "gpu_memory_utilization": 0.95},
]

print("GPU Memory Utilization 테스트 설정:")
for cfg in gpu_util_test_configs:
    print(f"  {cfg['name']}: {cfg['gpu_memory_utilization']}")

In [ ]:
# 테스트 실행 함수 (개념 코드)

def test_gpu_utilization(util_value):
    """
    특정 GPU 메모리 사용률로 테스트
    
    실제 실행 시:
    1. LLM 인스턴스 생성
    2. 여러 요청 처리
    3. 처리량/지연시간 측정
    4. OOM 발생 여부 확인
    """
    test_code = f'''
from vllm import LLM, SamplingParams
import time

# 모델 로드
llm = LLM(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    quantization="awq",
    gpu_memory_utilization={util_value}
)

# 테스트 실행
prompts = ["테스트 프롬프트"] * 100
sampling_params = SamplingParams(max_tokens=128)

start = time.time()
outputs = llm.generate(prompts, sampling_params)
elapsed = time.time() - start

print(f"처리량: {{len(prompts) / elapsed:.2f}} req/s")
'''
    return test_code

print("테스트 코드 예시 (gpu_memory_utilization=0.90):")
print(test_gpu_utilization(0.90))

## 3. Batch Size 튜닝

배치 크기 관련 파라미터 튜닝을 실험합니다.

In [ ]:
# Batch Size 관련 파라미터

batch_configs = [
    {
        "name": "Low Latency",
        "max_num_batched_tokens": 4096,
        "max_num_seqs": 64,
        "expected": "낮은 지연시간, 낮은 처리량"
    },
    {
        "name": "Balanced",
        "max_num_batched_tokens": 8192,
        "max_num_seqs": 128,
        "expected": "균형 잡힌 성능"
    },
    {
        "name": "High Throughput",
        "max_num_batched_tokens": 16384,
        "max_num_seqs": 256,
        "expected": "높은 처리량, 높은 지연시간"
    }
]

print("Batch Size 설정 비교:")
print("=" * 60)
for cfg in batch_configs:
    print(f"\n{cfg['name']}:")
    print(f"  max_num_batched_tokens: {cfg['max_num_batched_tokens']}")
    print(f"  max_num_seqs: {cfg['max_num_seqs']}")
    print(f"  예상: {cfg['expected']}")

In [ ]:
# Batch Size vs Latency 트레이드오프 시각화

import matplotlib.pyplot as plt
import numpy as np

# 예상 데이터 (실제 테스트 후 업데이트)
max_seqs = [32, 64, 128, 256, 512]
throughput = [15, 28, 42, 55, 60]  # 예상 req/s
latency_p50 = [80, 100, 130, 180, 250]  # 예상 ms
latency_p99 = [150, 200, 300, 450, 700]  # 예상 ms

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Throughput
axes[0].plot(max_seqs, throughput, 'b-o', linewidth=2, markersize=8)
axes[0].set_xlabel('max_num_seqs')
axes[0].set_ylabel('Throughput (req/s)')
axes[0].set_title('max_num_seqs vs Throughput')
axes[0].grid(True, alpha=0.3)

# Latency
axes[1].plot(max_seqs, latency_p50, 'g-o', linewidth=2, markersize=8, label='P50')
axes[1].plot(max_seqs, latency_p99, 'r-o', linewidth=2, markersize=8, label='P99')
axes[1].set_xlabel('max_num_seqs')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('max_num_seqs vs Latency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../benchmarks/results/batch_size_tradeoff.png', dpi=150)
plt.show()

print("그래프 저장: benchmarks/results/batch_size_tradeoff.png")

## 4. 양자화 방식 비교

다양한 양자화 방식에 따른 성능을 비교합니다.

In [ ]:
# 양자화 방식 비교

quantization_comparison = {
    "FP16 (No Quantization)": {
        "memory": "100%",
        "speed": "1.0x (baseline)",
        "quality": "100%",
        "7B model VRAM": "~14GB"
    },
    "AWQ (4-bit)": {
        "memory": "~25%",
        "speed": "~1.8x",
        "quality": "~99%",
        "7B model VRAM": "~4GB"
    },
    "GPTQ (4-bit)": {
        "memory": "~25%",
        "speed": "~1.5x",
        "quality": "~98%",
        "7B model VRAM": "~4GB"
    },
    "INT8": {
        "memory": "~50%",
        "speed": "~2.2x",
        "quality": "~95%",
        "7B model VRAM": "~7GB"
    }
}

print("양자화 방식 비교:")
print("=" * 80)
print(f"{'Method':<20} {'Memory':<10} {'Speed':<15} {'Quality':<10} {'7B VRAM':<10}")
print("-" * 80)

for method, info in quantization_comparison.items():
    print(f"{method:<20} {info['memory']:<10} {info['speed']:<15} {info['quality']:<10} {info['7B model VRAM']:<10}")

print("\n권장: AWQ (정확도 유지 + 속도 향상)")

## 5. 최적 설정 찾기

환경에 맞는 최적 설정을 결정합니다.

In [ ]:
# GPU 메모리별 권장 설정

def get_recommended_config(gpu_memory_gb):
    """GPU 메모리에 따른 권장 설정 반환"""
    
    if gpu_memory_gb <= 16:
        return {
            "gpu_memory_utilization": 0.85,
            "max_num_batched_tokens": 4096,
            "max_num_seqs": 64,
            "quantization": "awq",
            "note": "16GB 이하: AWQ 양자화 필수, 보수적 설정"
        }
    elif gpu_memory_gb <= 24:
        return {
            "gpu_memory_utilization": 0.90,
            "max_num_batched_tokens": 8192,
            "max_num_seqs": 128,
            "quantization": "awq",
            "note": "24GB: AWQ 양자화 권장, 균형 잡힌 설정"
        }
    elif gpu_memory_gb <= 48:
        return {
            "gpu_memory_utilization": 0.90,
            "max_num_batched_tokens": 16384,
            "max_num_seqs": 256,
            "quantization": "awq",  # 또는 FP16
            "note": "48GB: AWQ 또는 FP16, 고성능 설정"
        }
    else:
        return {
            "gpu_memory_utilization": 0.90,
            "max_num_batched_tokens": 32768,
            "max_num_seqs": 512,
            "quantization": "none",  # FP16 가능
            "note": "80GB+: FP16 가능, 최대 성능 설정"
        }

# GPU 메모리별 권장 설정 출력
gpu_memories = [16, 24, 48, 80]

print("GPU 메모리별 권장 설정:")
print("=" * 60)

for gpu_mem in gpu_memories:
    config = get_recommended_config(gpu_mem)
    print(f"\n{gpu_mem}GB GPU:")
    for key, value in config.items():
        print(f"  {key}: {value}")

In [ ]:
# 최적 설정 YAML 생성

def generate_optimal_config(gpu_memory_gb, use_case="balanced"):
    """최적 설정 YAML 생성"""
    
    base_config = get_recommended_config(gpu_memory_gb)
    
    # Use case 별 조정
    if use_case == "low_latency":
        base_config["max_num_seqs"] = max(32, base_config["max_num_seqs"] // 2)
        base_config["note"] = "저지연 최적화"
    elif use_case == "high_throughput":
        base_config["max_num_seqs"] = min(512, base_config["max_num_seqs"] * 2)
        base_config["note"] = "처리량 최적화"
    
    config_yaml = f"""
# vLLM Optimized Config
# GPU Memory: {gpu_memory_gb}GB
# Use Case: {use_case}

model:
  name: "Qwen/Qwen2.5-7B-Instruct-AWQ"
  quantization: "{base_config['quantization']}"

engine:
  gpu_memory_utilization: {base_config['gpu_memory_utilization']}
  max_num_batched_tokens: {base_config['max_num_batched_tokens']}
  max_num_seqs: {base_config['max_num_seqs']}
  block_size: 16

# Note: {base_config['note']}
"""
    return config_yaml

# RTX 4090 (24GB) 최적 설정 예시
print("RTX 4090 (24GB) 균형 잡힌 설정:")
print(generate_optimal_config(24, "balanced"))

## 요약

### 핵심 튜닝 포인트
1. **gpu_memory_utilization**: 0.90 권장 (프로덕션)
2. **max_num_seqs**: 처리량 vs 지연시간 트레이드오프
3. **양자화**: AWQ가 가장 효율적

### 다음 단계
- 실제 벤치마크 실행: `benchmarks/` 스크립트 활용
- 결과 분석: `03_comparison.ipynb`